In [3]:
import numpy as np
from lava.proc.lif.process import LIF
from lava.proc.io.sink import RingBuffer as Sink
from lava.magma.core.run_configs import Loihi2SimCfg
from lava.magma.core.run_conditions import RunSteps

# Create a single LIF neuron
# vth = threshold, dv = voltage decay, du = current decay
lif_neuron = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=10) # Changing bias_mant=5 (33 spikes) to bias_mant=10 (50 spikes)

# Create a sink to record the neuron's spike output
sink = Sink(shape=(1,), buffer=100)

# Connect the neuron's spike output to the sink
lif_neuron.s_out.connect(sink.a_in)

# Run for 100 timesteps
lif_neuron.run(condition=RunSteps(num_steps=100), 
               run_cfg=Loihi2SimCfg())

# Get the recorded spike data
spike_data = sink.data.get()

lif_neuron.stop()

print(f"Spike data shape: {spike_data.shape}")
print(f"Total spikes: {np.sum(spike_data)}")
print(f"Spike train: {spike_data.flatten()}")

Spike data shape: (1, 100)
Total spikes: 50.0
Spike train: [0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1.
 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1.
 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1.
 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1.
 0. 1. 0. 1.]


## Lava LIF Neuron — Observations & Results

### What is Lava?
Intel's open-source neuromorphic computing framework, designed specifically 
for Loihi hardware. Unlike Brian2's continuous differential equation approach, 
Lava operates on **discrete timesteps**, mirroring how real neuromorphic 
chips process information in clock cycles.

### Architecture
A single LIF neuron process with:
- `vth` — firing threshold
- `dv` — voltage decay rate (0 = no decay)
- `du` — current decay rate (0 = no decay)  
- `bias_mant` — constant current injected every timestep

### Key finding 1: Bias controls firing rate
| `bias_mant` | Total spikes (100 steps) | Pattern |
|-------------|--------------------------|---------|
| 5 | 33 | `0, 0, 1` repeating (fires every 3rd step) |
| 10 | 50 | `0, 1` repeating (fires every 2nd step) |

With `vth=10`, a bias of 5 requires 2 accumulation steps to cross threshold 
(fires on the 3rd). A bias of 10 crosses threshold every single step in 
principle, but actually fires every other step due to Lava's internal timing.

### Key finding 2: One-step processing delay
Initially expected `bias_mant=10` to produce 100/100 spikes (firing every 
step), but observed alternating `0, 1` pattern instead (50 spikes).

This reveals that **Lava's discrete simulation has a one-step processing 
delay** — the threshold check at each timestep uses the membrane state 
*before* that step's bias is fully integrated, rather than checking 
instantaneously like Brian2's continuous equations.

**Significance:** This is a framework-specific timing convention that 
matters when working with real neuromorphic hardware as Loihi 2 operates 
on actual clock cycles, so understanding these discrete-time subtleties 
is directly relevant to real hardware behaviour, not just a simulation quirk.